# Data Exploration: Customer Data Pipeline

## Scenario: Financial Reporting

Our CFO wants accurate monthly sales reports. Before we can report, we need to understand what's in our data and identify quality issues.

**The Problem:**
- Raw transaction data comes from our e-commerce application
- Applications optimize for speed, not data quality
- Reports need to be accurate — executives make decisions based on them

**Our Task:**
1. Profile the data (what do we have?)
2. Identify quality issues (what's wrong?)
3. Document findings (input for schema contract)

---

## Setup

In [ ]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

## 1. Load and Profile Customer Data

Let's start with the customer data. This represents our customer master data.

In [ ]:
# Load customer data
customers = pd.read_csv('../data/raw/customers.csv')

print(f"Shape: {customers.shape[0]} rows, {customers.shape[1]} columns")
print(f"\nColumns: {list(customers.columns)}")

In [ ]:
# Basic info
customers.info()

In [ ]:
# First few rows
customers.head(10)

### Customer Data Quality Check

In [ ]:
# Check for nulls
print("=== Null Values ===")
null_counts = customers.isnull().sum()
null_pct = (customers.isnull().sum() / len(customers) * 100).round(1)
null_df = pd.DataFrame({'Nulls': null_counts, 'Percent': null_pct})
print(null_df[null_df['Nulls'] > 0])

In [ ]:
# Check for duplicates
print("=== Duplicate Check ===")
dup_ids = customers[customers['customer_id'].duplicated(keep=False)]
print(f"Duplicate customer_ids: {len(dup_ids)} rows")
if len(dup_ids) > 0:
    print("\nDuplicate records:")
    print(dup_ids)

In [ ]:
# Check age values
print("=== Age Value Check ===")
print(f"Min age: {customers['age'].min()}")
print(f"Max age: {customers['age'].max()}")
print(f"\nSuspicious ages (< 0 or > 120):")
invalid_ages = customers[(customers['age'] < 0) | (customers['age'] > 120)]
print(invalid_ages[['customer_id', 'first_name', 'last_name', 'age']])

In [ ]:
# Check date formats
print("=== Date Format Check ===")
print("Sample signup_date values:")
print(customers['signup_date'].head(20).tolist())

# Identify different formats
iso_format = customers['signup_date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
us_format = customers['signup_date'].str.match(r'^\d{2}/\d{2}/\d{4}$', na=False)
print(f"\nISO format (YYYY-MM-DD): {iso_format.sum()} rows")
print(f"US format (MM/DD/YYYY): {us_format.sum()} rows")

### 🔍 Customer Data Quality Summary

**Issues Found:**
- [ ] **HIGH:** Duplicate customer_id — data integrity issue
- [ ] **MEDIUM:** ~16% null emails — decide if required
- [ ] **HIGH:** Invalid ages (-5, 999, 150) — need range validation
- [ ] **MEDIUM:** Mixed date formats — need standardization

---

## 2. Load and Profile Transaction Data

Transactions are the core of our sales reporting.

In [ ]:
# Load transaction data
transactions = pd.read_csv('../data/raw/transactions.csv')

print(f"Shape: {transactions.shape[0]} rows, {transactions.shape[1]} columns")
print(f"\nColumns: {list(transactions.columns)}")

In [ ]:
# Basic info
transactions.info()

In [ ]:
# First few rows
transactions.head(10)

### Transaction Data Quality Check

In [ ]:
# Check for nulls
print("=== Null Values ===")
null_counts = transactions.isnull().sum()
null_pct = (transactions.isnull().sum() / len(transactions) * 100).round(1)
null_df = pd.DataFrame({'Nulls': null_counts, 'Percent': null_pct})
print(null_df[null_df['Nulls'] > 0])

In [ ]:
# Check for orphan customer_ids (transactions for non-existent customers)
print("=== Orphan Customer Check ===")
valid_customers = set(customers['customer_id'].dropna())
txn_customers = set(transactions['customer_id'].dropna())

orphan_customers = txn_customers - valid_customers
print(f"Customer IDs in transactions but not in customers: {orphan_customers}")

orphan_txns = transactions[transactions['customer_id'].isin(orphan_customers)]
print(f"\nOrphan transactions: {len(orphan_txns)} rows")
if len(orphan_txns) > 0:
    print(orphan_txns)

In [ ]:
# Check amount values
print("=== Amount Value Check ===")
print(f"Min amount: ${transactions['amount'].min():.2f}")
print(f"Max amount: ${transactions['amount'].max():.2f}")
print(f"Mean amount: ${transactions['amount'].mean():.2f}")

# Negative amounts (refunds?)
negative = transactions[transactions['amount'] < 0]
print(f"\nNegative amounts: {len(negative)} rows")
if len(negative) > 0:
    print(negative)

In [ ]:
# Check date formats (same issue as customers)
print("=== Date Format Check ===")
iso_format = transactions['transaction_date'].str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
us_format = transactions['transaction_date'].str.match(r'^\d{2}/\d{2}/\d{4}$', na=False)
print(f"ISO format (YYYY-MM-DD): {iso_format.sum()} rows")
print(f"US format (MM/DD/YYYY): {us_format.sum()} rows")

### 🔍 Transaction Data Quality Summary

**Issues Found:**
- [ ] **HIGH:** Null customer_id — can't attribute sales
- [ ] **HIGH:** Orphan customer_id (C999) — referential integrity issue
- [ ] **MEDIUM:** Negative amounts — are these refunds? Need business rule
- [ ] **MEDIUM:** Mixed date formats — same as customers

---

## 3. Impact on Reporting

Let's see how these quality issues affect our sales report.

In [ ]:
# Raw total sales (what we'd report without cleaning)
raw_total = transactions['amount'].sum()
print(f"=== Raw Sales Total ===")
print(f"Total: ${raw_total:,.2f}")

In [ ]:
# What if we exclude problematic records?
print("=== Impact of Quality Issues ===")

# Exclude null customer_ids
valid_txns = transactions[transactions['customer_id'].notna()]
print(f"After removing null customer_id: ${valid_txns['amount'].sum():,.2f}")

# Exclude orphan customer_ids
valid_txns = valid_txns[~valid_txns['customer_id'].isin(orphan_customers)]
print(f"After removing orphan customers: ${valid_txns['amount'].sum():,.2f}")

# Exclude negative amounts (treat as separate refund report)
positive_txns = valid_txns[valid_txns['amount'] >= 0]
print(f"After excluding negative amounts: ${positive_txns['amount'].sum():,.2f}")

print(f"\n=== Difference ===")
print(f"Raw total: ${raw_total:,.2f}")
print(f"Clean total: ${positive_txns['amount'].sum():,.2f}")
print(f"Difference: ${raw_total - positive_txns['amount'].sum():,.2f}")

### 💡 Key Insight

The difference between raw and clean totals shows why data platforms matter:

- **Applications** give us fast access to current data
- **Data platforms** give us accurate data for decisions

If the CFO used raw data, the report would be wrong.

---

## 4. Next Steps

Based on this exploration, we need to:

1. **Create a schema contract** defining validation rules for each column
2. **Implement validation** to catch these issues automatically
3. **Write transformations** to clean the data
4. **Document decisions** (e.g., how to handle negative amounts)

### Questions for data-advisor:

- How should we handle the orphan customer_id (C999)? Reject the transactions or create a placeholder customer?
- Should negative amounts be excluded from sales reports or treated as refunds in a separate report?
- For invalid ages, should we reject the record, set to null, or use a default?

### Document in HISTORY.md:

```markdown
## Exploration: Raw Data Quality

### Findings
- Customer duplicates: 1 (C001)
- Null emails: 16%
- Invalid ages: 3 records
- Orphan transactions: 1 (C999)
- Mixed date formats in both tables

### Decisions Needed
- [ ] Email: required or optional?
- [ ] Age: reject invalid or set to null?
- [ ] Orphan customer_id: reject or placeholder?
- [ ] Negative amounts: exclude or separate report?

### Next
- Discuss with data-advisor
- Create schema contract proposal
```